# VisClick — Colab setup

Run cells **top to bottom**. Authorise Google Drive when prompted.

- **GitHub** → code at `/content/visclick`
- **Drive** `MyDrive/visclick` → datasets + weights (persists after disconnect)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
if not os.path.isdir("/content/visclick"):
    subprocess.run(["git", "clone", REPO, "/content/visclick"], check=True)
else:
    subprocess.run("cd /content/visclick && git pull --rebase", shell=True, check=False)
%cd /content/visclick


In [ ]:
import subprocess
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
        print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print("GPU check:", e)


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
os.makedirs(DRIVE, exist_ok=True)
for sub in (
    "data/raw", "data/clay", "data/vins", "data/webui", "data/ui_vision",
    "data/unified", "data/source_train", "data/desktop_labeled", "data/desktop_test",
    "weights/baseline_source", "weights/experiments", "videos",
):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print("DRIVE =", DRIVE)


In [ ]:
!pip install -q ultralytics==8.* opencv-python "huggingface_hub[cli]>=0.20" tqdm matplotlib pandas


## 02 — Train source domain (YOLOv8s)

Requires `DRIVE/.../data/source_train` in YOLO layout with `images/{train,val,test}` and `labels/{...}` and `yolo_source_colab.yaml` (generated by patch script). **Edit `DRIVE` if your folder name differs.**


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
os.environ["VISCLICK_DRIVE"] = DRIVE
!python scripts/patch_colab_configs.py


In [ ]:
# Verify dataset path exists
import os
DRIVE = "/content/drive/MyDrive/visclick"
st = os.path.join(DRIVE, "data", "source_train")
ok = os.path.isdir(os.path.join(st, "images", "train")) and os.path.isdir(os.path.join(st, "labels", "train"))
print("source_train OK:", ok, st)
if not ok:
    print("Build source_train (see plan C.5) or point Zenodo unified to YOLO + map classes to 6 before training here.")


In [ ]:
# Train (writes weights to Drive)
import time
from ultralytics import YOLO

DRIVE = "/content/drive/MyDrive/visclick"
t0 = time.time()
model = YOLO("yolov8s.pt")
results = model.train(
    data="configs/yolo_source_colab.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    workers=2,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    patience=10,
    project=f"{DRIVE}/weights/baseline_source",
    name="run1",
    seed=0,
    save=True,
    plots=True,
)
print("train wall (s):", int(time.time() - t0))


In [ ]:
# Copy best weights to stable name
import os, shutil
DRIVE = "/content/drive/MyDrive/visclick"
src = f"{DRIVE}/weights/baseline_source/run1/weights/best.pt"
dst = f"{DRIVE}/weights/baseline_source/best_source_v8s.pt"
if os.path.isfile(src):
    shutil.copy2(src, dst)
    print("Saved", dst)
else:
    print("Missing", src, "- finish training or resume from last.pt")


In [ ]:
# Validate + metrics
from ultralytics import YOLO
import os
DRIVE = "/content/drive/MyDrive/visclick"
p = f"{DRIVE}/weights/baseline_source/best_source_v8s.pt"
if not os.path.isfile(p):
    p = f"{DRIVE}/weights/baseline_source/run1/weights/best.pt"
m = YOLO(p)
metrics = m.val(data="configs/yolo_source_colab.yaml", split="test")
print(metrics)
